## Parameter Management

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(), nn.LazyLinear(1))
X = torch.rand(size=(2, 4))
net(X).shape, " ",net(X) 

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


(torch.Size([2, 1]),
 ' ',
 tensor([[0.3022],
         [0.2906]], grad_fn=<AddmmBackward0>))

### Parameter Access

In [3]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[ 0.0985, -0.1438, -0.0690,  0.1831, -0.2504,  0.1377,  0.3095, -0.2926]])),
             ('bias', tensor([0.3181]))])

In [4]:
net[0].state_dict(), net[1].state_dict()

(OrderedDict([('weight',
               tensor([[ 0.2600, -0.4302, -0.1440,  0.2544],
                       [ 0.4303, -0.1160, -0.3669,  0.4109],
                       [-0.2324, -0.4470,  0.2511, -0.3772],
                       [-0.1088,  0.2504, -0.0248, -0.4214],
                       [-0.1913, -0.0898, -0.3207,  0.3500],
                       [ 0.4873,  0.1412,  0.2270,  0.2144],
                       [ 0.1407,  0.3678,  0.0326, -0.3900],
                       [-0.3443, -0.1872,  0.2842,  0.1896]])),
              ('bias',
               tensor([-0.3076, -0.2121,  0.4383, -0.0427,  0.0623, -0.2691, -0.0613, -0.3332]))]),
 OrderedDict())

### Targeted Parameters

In [5]:
type(net[2].bias), net[2].bias.data

(torch.nn.parameter.Parameter, tensor([0.3181]))

In [6]:
type(net[2].weight), net[2].weight.data

(torch.nn.parameter.Parameter,
 tensor([[ 0.0985, -0.1438, -0.0690,  0.1831, -0.2504,  0.1377,  0.3095, -0.2926]]))

In [7]:
net[2].weight.grad == None

True

### All parameters at Once

In [8]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([1, 8])),
 ('2.bias', torch.Size([1]))]

In [9]:
list(net.named_parameters())

[('0.weight',
  Parameter containing:
  tensor([[ 0.2600, -0.4302, -0.1440,  0.2544],
          [ 0.4303, -0.1160, -0.3669,  0.4109],
          [-0.2324, -0.4470,  0.2511, -0.3772],
          [-0.1088,  0.2504, -0.0248, -0.4214],
          [-0.1913, -0.0898, -0.3207,  0.3500],
          [ 0.4873,  0.1412,  0.2270,  0.2144],
          [ 0.1407,  0.3678,  0.0326, -0.3900],
          [-0.3443, -0.1872,  0.2842,  0.1896]], requires_grad=True)),
 ('0.bias',
  Parameter containing:
  tensor([-0.3076, -0.2121,  0.4383, -0.0427,  0.0623, -0.2691, -0.0613, -0.3332],
         requires_grad=True)),
 ('2.weight',
  Parameter containing:
  tensor([[ 0.0985, -0.1438, -0.0690,  0.1831, -0.2504,  0.1377,  0.3095, -0.2926]],
         requires_grad=True)),
 ('2.bias',
  Parameter containing:
  tensor([0.3181], requires_grad=True))]

### Tied Parameters

In [10]:
# We need to give the shared layer a name so that we can refer to its
# parameters
shared = nn.LazyLinear(8)
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.LazyLinear(1))
net(X)

tensor([[0.1268],
        [0.1267]], grad_fn=<AddmmBackward0>)

In [11]:
# Check whether the parameters are the same
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100

tensor([True, True, True, True, True, True, True, True])


In [12]:
# Make sure that they are actually the same object rather than just having the
# same value
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])


## Exercises

### Ex. 1

In [13]:
X = torch.rand(2, 20)

In [14]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Random weight parameters that will not compute gradients and
        # therefore keep constant during training
        self.rand_weight = torch.rand((20, 20))
        self.linear = nn.LazyLinear(20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(X @ self.rand_weight + 1)
        # Reuse the fully connected layer. This is equivalent to sharing
        # parameters with two fully connected layers
        X = self.linear(X)
        # Control flow
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [15]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.LazyLinear(64), nn.ReLU(),
                                 nn.LazyLinear(32), nn.ReLU())
        self.linear = nn.LazyLinear(16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.LazyLinear(20), FixedHiddenMLP())
chimera(X)

tensor(-0.0162, grad_fn=<SumBackward0>)

In [16]:
list(chimera.named_parameters())

[('0.net.0.weight',
  Parameter containing:
  tensor([[ 0.0665, -0.1003,  0.0265,  ...,  0.0655, -0.0314,  0.2061],
          [-0.0390, -0.0414,  0.1310,  ..., -0.1829, -0.0478,  0.1757],
          [ 0.0869, -0.1195,  0.0764,  ..., -0.0665,  0.0882,  0.2174],
          ...,
          [ 0.0027, -0.0814, -0.1852,  ...,  0.2133, -0.2057,  0.1662],
          [-0.1220,  0.2231,  0.0405,  ..., -0.1229,  0.1834, -0.1949],
          [ 0.1338, -0.1374, -0.0958,  ...,  0.0670, -0.1596,  0.0975]],
         requires_grad=True)),
 ('0.net.0.bias',
  Parameter containing:
  tensor([-0.1273, -0.0271, -0.0476, -0.0638,  0.2125,  0.1951,  0.0790,  0.2067,
           0.0742, -0.0039, -0.0316,  0.0175, -0.1133,  0.2146, -0.0878, -0.2114,
          -0.0201, -0.2057, -0.0031, -0.0196,  0.0688, -0.1645, -0.1675, -0.0541,
          -0.2202,  0.0536,  0.0527, -0.0597,  0.0490,  0.1018, -0.0876, -0.1813,
           0.1458, -0.0747,  0.0025, -0.0835,  0.0562,  0.1470,  0.2106,  0.0339,
           0.0797, -0.036

In [17]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([8, 8])),
 ('2.bias', torch.Size([8])),
 ('6.weight', torch.Size([1, 8])),
 ('6.bias', torch.Size([1]))]

### Ex. 2

In [27]:
X = torch.rand(size=(4, 4))
X, X.shape

(tensor([[0.2626, 0.6933, 0.1505, 0.0816],
         [0.7134, 0.6085, 0.8803, 0.1084],
         [0.6984, 0.6613, 0.4504, 0.2003],
         [0.2636, 0.2370, 0.8491, 0.2779]]),
 torch.Size([4, 4]))

In [33]:
shared_layer = nn.LazyLinear(8)
shared_net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared_layer, nn.ReLU(),
                    shared_layer, nn.ReLU(),
                    nn.LazyLinear(1))
shared_net(X)

tensor([[-0.2689],
        [-0.2678],
        [-0.2678],
        [-0.2519]], grad_fn=<AddmmBackward0>)

In [36]:
[(name, param.shape) for name, param in shared_net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([8, 8])),
 ('2.bias', torch.Size([8])),
 ('6.weight', torch.Size([1, 8])),
 ('6.bias', torch.Size([1]))]

In [48]:
shared_net[2].weight == shared_net[4].weight, shared_net[2].bias == shared_net[4].bias

(tensor([[True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True],
         [True, True, True, True, True, True, True, True]]),
 tensor([True, True, True, True, True, True, True, True]))

In [44]:
shared_net[0].weight, shared_net[2].weight, shared_net[6].weight

(Parameter containing:
 tensor([[-0.2272,  0.2848, -0.4210,  0.0029],
         [-0.3046,  0.4466,  0.1086,  0.4410],
         [ 0.0016,  0.1648, -0.4730,  0.3519],
         [ 0.2302,  0.4278,  0.1239, -0.2583],
         [-0.1648, -0.1095, -0.2136,  0.3620],
         [-0.3677, -0.3063,  0.1001,  0.2936],
         [-0.2199, -0.2947,  0.1272,  0.2697],
         [ 0.4883, -0.2236, -0.3988, -0.3930]], requires_grad=True),
 Parameter containing:
 tensor([[ 0.1560, -0.3477, -0.0232, -0.2913, -0.0485, -0.0338,  0.2257, -0.0193],
         [-0.0970,  0.3321,  0.0464, -0.0301, -0.2423,  0.0108, -0.0751,  0.1998],
         [ 0.2287,  0.2647,  0.2126,  0.3295,  0.1302, -0.0871, -0.2326,  0.1932],
         [-0.0139, -0.0153, -0.1524, -0.0873,  0.0551,  0.0079,  0.1814, -0.0030],
         [-0.2782,  0.0745, -0.0520, -0.0780,  0.1846,  0.3341,  0.0859,  0.1124],
         [-0.3342, -0.2242,  0.3504,  0.1805, -0.1883,  0.0747,  0.2845,  0.0233],
         [-0.1987, -0.1080,  0.0911, -0.2156,  0.0064,  0.

In [49]:
class SharedNet(nn.Module):
    def __init__(self):
        super().__init__()
        shared_layer = nn.LazyLinear(8)
        self.net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared_layer, nn.ReLU(),
                    shared_layer, nn.ReLU(),
                    nn.LazyLinear(1))
        
    def forward(self, X):
        return self.net(X)

In [60]:
Shared_Net = SharedNet()
optim = torch.optim.SGD(Shared_Net.parameters(), lr=0.1)
loss = nn.MSELoss()

y_true = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

C:\Users\hub30\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [61]:
for epoch in range(10):
    y_pred = Shared_Net(X)
    l = loss(y_pred, y_true)
    optim.zero_grad()
    l.backward()
    
    shared_layer = Shared_Net.net[2] 
    grad_norm = shared_layer.weight.grad.norm().item()
    weight_sample = shared_layer.weight[0, 0].item()    
    print(f"Epoch {epoch+1} | Loss: {l.item():.4f} | Grad Norm: {grad_norm:.4f} | Weight[0,0]: {weight_sample:.4f}")
    optim.step()

Epoch 1 | Loss: 0.3905 | Grad Norm: 0.2429 | Weight[0,0]: -0.2960
Epoch 2 | Loss: 0.3125 | Grad Norm: 0.1606 | Weight[0,0]: -0.2877
Epoch 3 | Loss: 0.2793 | Grad Norm: 0.0979 | Weight[0,0]: -0.2822
Epoch 4 | Loss: 0.2664 | Grad Norm: 0.0559 | Weight[0,0]: -0.2790
Epoch 5 | Loss: 0.2617 | Grad Norm: 0.0850 | Weight[0,0]: -0.2773
Epoch 6 | Loss: 0.2598 | Grad Norm: 0.0343 | Weight[0,0]: -0.2768
Epoch 7 | Loss: 0.2587 | Grad Norm: 0.0558 | Weight[0,0]: -0.2768
Epoch 8 | Loss: 0.2576 | Grad Norm: 0.0575 | Weight[0,0]: -0.2773
Epoch 9 | Loss: 0.2566 | Grad Norm: 0.0587 | Weight[0,0]: -0.2779
Epoch 10 | Loss: 0.2559 | Grad Norm: 0.0442 | Weight[0,0]: -0.2787
